# Rebuild Discount From Product Description

## Objective
Recompute the `Discount` value using promotional tags embedded in the `Description` column.

## Source Table
`cgdc_sales`

## Columns

| Column | Type |
|----------|----------|
| StoreID | String |
| ProductName | String |
| Category | String |
| SoldUnits | Integer |
| Description | String |
| Discount | Decimal/Double |

## Input Dataset
The table contains product sales records along with descriptions that may include promotional discount tags.

Example tag format:

[N% off]

Examples:

- [5% off]
- [10% off]
- [20% off]

## Expected Output Columns

- StoreID
- ProductName
- Category
- SoldUnits
- Description
- Discount

## Requirements

- Return every row from `cgdc_sales`.
- Recompute the `Discount` value from the promotion tag contained in `Description`.
- If no valid promotion tag exists, the discount should be treated as zero.
- Preserve all non-discount column values unchanged.
- Preserve original row ordering.
- Output column names must exactly match the required names.

## Sample Record

| StoreID | ProductName | Category | SoldUnits | Description | Discount |
|----------|----------|----------|----------|----------|----------|
| S101 | Biscuits | Food | 120 | Tasty Biscuits [10% off] | 0.0 |

In [0]:
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, DoubleType
from pyspark.sql.functions import *

cgdc_sales_schema = StructType([
    StructField("StoreID", StringType(), True),
    StructField("ProductName", StringType(), True),
    StructField("Category", StringType(), True),
    StructField("SoldUnits", IntegerType(), True),
    StructField("Description", StringType(), True),
    StructField("Discount", DoubleType(), True)
])

cgdc_sales_data = [
    ("S101", "Biscuits",   "Food",     120, "Tasty Biscuits [10% off]",      0.0),
    ("S102", "Shampoo",    "Hygiene",   85, "Smoothens Hair [5% off]",       0.25),
    ("S103", "Banana",     "Food",     150, "Fresh Bananas",                 0.0),
    ("S101", "Toothpaste", "Hygiene",  300, "Protects Teeth",                0.0),
    ("S102", "Shirt",      "Clothes",   65, "Cotton Shirts [20% off]",       0.0)
]

cgdc_sales_df = spark.createDataFrame(
    cgdc_sales_data,
    schema=cgdc_sales_schema
)

In [0]:
cgdc_sales_df = cgdc_sales_df.select(
    col("StoreID"),
    col("ProductName"),
    col("Category"),
    col("SoldUnits"),
    col("Description"),
    coalesce(
        round(regexp_substr(col("Description"), lit(r"\d+")) / 100, 2), lit(0.0)
    ).alias("Discount"),
)
display(cgdc_sales_df)